In [1]:
!pip install torch torchvision segment-anything-fast 

     ---------------------------------------- 19.7/19.7 MB 7.5 MB/s eta 0:00:00
     ------------------------------------- 612.0/612.0 kB 13.1 MB/s eta 0:00:00
     ---------------------------------------- 1.4/1.4 MB 9.7 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ------------------------------------- 178.7/178.7 kB 10.5 MB/s eta 0:00:00
     -------------------------------------- 117.0/117.0 kB 7.1 MB/s eta 0:00:00
     ---------------------------------------- 125.1/125.1 kB ? eta 0:00:00
     -------------------------------------- 616.3/616.3 kB 9.6 MB/s eta 0:00:00
     ---------------------------------------- 50.5/50.5 kB ? eta 0:00:00
     ---------------------------------------- 68.8/68.8 kB ? eta 0:00:00
     -------------------------------------- 456.7/456.7 kB 9.5 MB/s eta 0:00:00
     ---------------------------------------- 73.5/73.5 kB 4.2 MB/s eta 0:00:00
     ----------------------------------


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [18]:
import cv2
import numpy as np
from ultralytics import YOLO, SAM

# -------------------------------
# 1. MODELLERİ YÜKLE
# -------------------------------

yolo_model = YOLO(r"best2.pt")
sam_model = SAM(r"C:\Users\User\Desktop\MobileSAM\mobile_sam.pt")

# -------------------------------
# 2. GÖRSELİ YÜKLE
# -------------------------------

image_path = "gorsel.jpg"
img = cv2.imread(image_path)

if img is None:
    print("Görsel bulunamadı!")
    exit()

img_kaba = img.copy()
img_sam = img.copy()

# -------------------------------
# 3. YOLO İLE NESNE TESPİTİ
# -------------------------------

yolo_results = yolo_model(img)

boxes = yolo_results[0].boxes.xyxy.cpu().numpy()

if len(boxes) == 0:
    print("YOLO nesne bulamadı.")
    exit()

# İlk nesneyi al
x1, y1, x2, y2 = boxes[0].astype(int)

kullanici_kutusu = [x1, y1, x2, y2]

# -------------------------------
# 4. KABA BOX ÇİZ
# -------------------------------

cv2.rectangle(img_kaba, (x1, y1), (x2, y2), (255, 0, 0), 4)

cv2.putText(img_kaba, "YOLO Kaba Box",
            (x1, y1 - 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.2,
            (255, 0, 0),
            3)

# -------------------------------
# 5. SAM İLE RAFİNE ET
# -------------------------------

results = sam_model.predict(source=img, bboxes=[kullanici_kutusu])

if results[0].masks is not None:

    all_masks = results[0].masks.data.cpu().numpy()

    mask_areas = [np.sum(m) for m in all_masks]
    best_mask_idx = np.argmax(mask_areas)

    mask = all_masks[best_mask_idx]

    coords = np.argwhere(mask > 0)

    if len(coords) > 0:

        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)

        cv2.rectangle(img_sam,
                      (int(x_min), int(y_min)),
                      (int(x_max), int(y_max)),
                      (0, 255, 0),
                      4)

        cv2.putText(img_sam,
                    "SAM Rafine Box",
                    (int(x_min), int(y_min) - 15),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (0, 255, 0),
                    3)

        print("YOLO Box:", kullanici_kutusu)
        print("SAM Box:", [x_min, y_min, x_max, y_max])

        cv2.imshow("YOLO Kaba", img_kaba)
        cv2.imshow("SAM Rafine", img_sam)

        cv2.waitKey(0)

    else:
        print("SAM maske üretti ama boş.")

else:
    print("SAM nesne bulamadı.")

cv2.destroyAllWindows()



0: 640x448 1 sweater, 116.4ms
Speed: 6.1ms preprocess, 116.4ms inference, 16.9ms postprocess per image at shape (1, 3, 640, 448)

0: 1024x1024 1 0, 1549.3ms
Speed: 17.0ms preprocess, 1549.3ms inference, 1.8ms postprocess per image at shape (1, 3, 1024, 1024)
YOLO Box: [np.int64(15), np.int64(37), np.int64(166), np.int64(237)]
SAM Box: [np.int64(16), np.int64(36), np.int64(167), np.int64(236)]
